# Montandon: Earthquakes

This notebook fetches recent earthquake data from USGS via Montandon, the global crisis data bank, filtering for magnitude, and visualizes them on a map.

In [1]:
import os
import pandas as pd
from pystac_client import Client
import geopandas as gpd
from shapely.geometry import Point, Polygon, shape
from lonboard import viz

from datetime import datetime, timedelta, timezone

## Connect to Montandon STAC

Montandon is exposed as a STAC collection, but requires authentication.

In [2]:
STAC_API_URL = "https://montandon-eoapi.ifrc.org/stac"
API_TOKEN = os.getenv('MONTANDON_API_TOKEN')

In [3]:
auth_headers = {"Authorization": f"Bearer {API_TOKEN}"}

### Check Auth

In [4]:
# Connect to STAC API with authentication
try:
    client = Client.open(STAC_API_URL, headers=auth_headers)
    print(f"\n[OK] Connected to: {STAC_API_URL}")
    print(f"[OK] API Title: {client.title}")
    print(f"[OK] Authentication: Bearer Token (OpenID Connect)")
except Exception as e:
    print(f"\n[ERROR] Authentication failed: {e}")


[OK] Connected to: https://montandon-eoapi.ifrc.org/stac
[OK] API Title: Montandon STAC API
[OK] Authentication: Bearer Token (OpenID Connect)


In [5]:
client

<Client id=montandon-eoapi>

## Fetch most recent Events

List collections with event items, usig simple string-matching on the collection id

In [6]:
collections_list = list(client.get_collections())

In [7]:
events_collection = [c for c in collections_list if '-events' in c.id]

In [8]:
for col in events_collection:
    print(f"{col.title} | {col.id}")

DesInventar Mapped Events | desinventar-events
EM-DAT Source Events | emdat-events
GDACS Source Events | gdacs-events
GFD Source Events | gfd-events
GLIDE Source Events | glide-events
IBTrACS Source Events | ibtracs-events
IDMC GIDD Source Events | idmc-gidd-events
IDMC Internal Displacement Updates (IDU) Impacts | idmc-idu-events
IFRC Source Events | ifrcevent-events
PDC Source Events | pdc-events
USGS Events | usgs-events


### Earthquakes: Single Collection Search
Search a single collection for the most earthquakes from USGS that have a magnitude > 5 in the past week

In [9]:
now = datetime.now(timezone.utc)
one_week_ago = two_days_ago = now - timedelta(hours=168)
magnitude = 4 # minimum magnitude to search for

In [10]:
search_hazards = client.search(
    collections=["usgs-hazards"],
    datetime=f"{one_week_ago.isoformat()}/{now.isoformat()}",
    max_items=1000,
)

earthquake_hazards = [
    item for item in search_hazards.items()
    if item.properties.get("monty:hazard_detail", {}).get("severity_value", 0) > magnitude
]

In [11]:
earthquake_hazards[0]

<Item id=usgs-hazard-us6000sz43-shakemap>

In [12]:
# Get the shape of an item
len(earthquake_hazards)

72

In [13]:
# ShakeMap polygons are only generated for significant events near populated areas.
# For everything else, fetch the epicenter Point from usgs-events as a fallback.
corr_ids = {item.properties['monty:corr_id'] for item in earthquake_hazards}

search_events = client.search(
    collections=["usgs-events"],
    datetime=f"{one_week_ago.isoformat()}/{now.isoformat()}",
    max_items=1000,
)

epicenters = {
    item.properties['monty:corr_id']: shape(item.geometry)
    for item in search_events.items()
    if item.properties.get('monty:corr_id') in corr_ids
}
print(f"Epicenter Points found for {len(epicenters)} / {len(corr_ids)} earthquakes")

Epicenter Points found for 72 / 72 earthquakes


In [14]:
def item_geometry(item):
    """Return ShakeMap polygon if valid, else epicenter Point from usgs-events."""
    geom = shape(item.geometry)
    if not geom.is_empty and geom.bounds != (0.0, 0.0, 0.0, 0.0):
        return geom  # ShakeMap polygon
    bbox = item.bbox
    if bbox and not all(v == 0 for v in bbox):
        return Point((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)
    corr_id = item.properties.get('monty:corr_id')
    return epicenters.get(corr_id)  # epicenter Point, or None if missing from events

gdf = gpd.GeoDataFrame(
    [{
        'id': item.id,
        'monty_corr_id': item.properties['monty:corr_id'],
        'datetime': pd.to_datetime(item.properties['datetime']),
        'title': item.properties['title'],
        'magnitude': item.properties['eq:magnitude'],
        'geometry': item_geometry(item),
    } for item in earthquake_hazards],
    geometry='geometry',
    crs='EPSG:4326',
).sort_values('datetime', ascending=False)

null_geom = gdf['geometry'].isna().sum()
print(f"{len(gdf)} earthquakes total, {null_geom} with no usable geometry")

72 earthquakes total, 0 with no usable geometry


In [15]:
earthquakes = gdf
earthquakes

,id,monty_corr_id,datetime,title,magnitude,geometry
0,usgs-hazard-us6000sz43-shakemap,20260520-UNK-755753-GH0311-1-GCDB,2026-05-20 23:12:56.417000+00:00,"M 4.9 - 258 km WSW of Tual, Indonesia",4.9,POINT (130.4649 -6.1047)
1,usgs-hazard-us6000sz3x-shakemap,20260520-ARG-303160-GH0311-1-GCDB,2026-05-20 23:01:01.063000+00:00,M 5.7 - South Sandwich Islands region,5.7,"POLYGON ((-31.45 -57.9, -25.017 -57.9, -25.017..."
2,usgs-hazard-us6000sz1t-shakemap,20260520-UNK-317107-GH0311-1-GCDB,2026-05-20 18:36:39.632000+00:00,M 5.0 - southern East Pacific Rise,5.0,POINT (-118.6623 -54.7454)
3,usgs-hazard-us6000sz1d-shakemap,20260520-UNK-304489-GH0311-1-GCDB,2026-05-20 17:43:01.910000+00:00,M 6.6 - southern East Pacific Rise,6.6,"POLYGON ((-126 -57.917, -118.933 -57.917, -118..."
4,usgs-hazard-us6000sz0k-shakemap,20260520-UNK-635348-GH0311-1-GCDB,2026-05-20 16:53:34.093000+00:00,"M 4.5 - 25 km ENE of Isangel, Vanuatu",4.5,POINT (169.4819 -19.4154)
...,...,...,...,...,...,...
67,usgs-hazard-us6000syc2-shakemap,20260517-UNK-1268887-GH0311-1-GCDB,2026-05-17 08:04:17.461000+00:00,"M 5.4 - 84 km ENE of Severo-Kuril’sk, Russia",5.4,"POLYGON ((154.567 49.017, 160.25 49.017, 160.2..."
68,usgs-hazard-us6000syby-shakemap,20260517-UNK-784640-GH0311-1-GCDB,2026-05-17 07:51:56.627000+00:00,"M 5.1 - 109 km SSE of Lorengau, Papua New Guinea",5.1,POINT (147.7681 -2.8876)
69,usgs-hazard-us6000sybb-shakemap,20260517-UNK-453595-GH0311-1-GCDB,2026-05-17 03:20:37.916000+00:00,"M 4.5 - 132 km SSE of Wainui, New Zealand",4.5,POINT (178.891 -39.6956)
70,usgs-hazard-us6000syb7-shakemap,20260517-UNK-524343-GH0311-1-GCDB,2026-05-17 02:41:41.290000+00:00,"M 4.2 - 28 km WSW of Illapel, Chile",4.2,POINT (-71.4663 -31.6854)


In [16]:
print(f"There have been {len(earthquakes)} recent earthquakes with a magnitude > 5, with {len(earthquakes['monty_corr_id'].unique())} Montandon correlation IDs.")

There have been 72 recent earthquakes with a magnitude > 5, with 72 Montandon correlation IDs.


In [17]:
from lonboard import Map, PolygonLayer, ScatterplotLayer
import numpy as np

# Split by geometry type: ShakeMap polygons for large events, epicenter Points for the rest
poly_gdf = earthquakes[earthquakes.geometry.geom_type == 'Polygon'].copy()
point_gdf = earthquakes[earthquakes.geometry.geom_type == 'Point'].copy()

# Scale radius by magnitude — each unit doubles the circle, matching the log scale of magnitude.
# M4 → ~20 km, M5 → ~40 km, M6 → ~80 km, M6.6 → ~122 km
radii = (2 ** (point_gdf['magnitude'] - 4) * 20_000).round().astype(int).values

layers = []
if len(poly_gdf):
    layers.append(PolygonLayer.from_geopandas(
        poly_gdf,
        get_fill_color=[220, 80, 0, 120],
        get_line_color=[180, 40, 0, 220],
    ))
if len(point_gdf):
    layers.append(ScatterplotLayer.from_geopandas(
        point_gdf,
        get_radius=radii,
        get_fill_color=[220, 80, 0, 180],
        radius_min_pixels=3,
    ))

print(f"{len(poly_gdf)} ShakeMap polygons, {len(point_gdf)} epicenter points")
Map(layers=layers, show_tooltip=True)

11 ShakeMap polygons, 61 epicenter points
